# FinGPT → MedicalGPT：最小可复用 SFT + DPO 训练流程（Notebook）

目标：用 **FinGPT 数据内容** + **MedicalGPT 多阶段方法** 跑通：
1. FinGPT 数据下载与本地落盘
2. 转换为 MedicalGPT 的 SFT / DPO 格式（符合 `docs/datasets.md`）
3. SFT 训练（LoRA）
4. DPO 偏好数据构建与 DPO 训练（LoRA）


## 0. 环境准备（可选）

如果你在全新环境运行，请先安装依赖；若已按项目 README 配置好环境可跳过。


## 1. 配置参数（Qwen2.5-7B + fingpt-sentiment-train）

本 Notebook 以 `FinGPT/fingpt-sentiment-train` 为示例，便于先完成“可跑通”的端到端流程。


In [ ]:
# HF Mirror（可选，国内网络推荐）
HF_ENDPOINT = "https://hf-mirror.com"


In [ ]:
import os
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


In [ ]:
from pathlib import Path

# ===== 可调参数 =====
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
FIN_DATASET = "FinGPT/fingpt-sentiment-train"
FIN_SPLIT = "train"

OUT_DIR = Path("data/fingpt_min")
RAW_DIR = OUT_DIR / "raw"
SFT_DIR = OUT_DIR / "sft"
DPO_DIR = OUT_DIR / "dpo"

RAW_FILE = RAW_DIR / "fingpt_raw.jsonl"
SFT_FILE = SFT_DIR / "fingpt_sft_sharegpt.jsonl"
DPO_FILE = DPO_DIR / "fingpt_dpo_pairs.jsonl"

SFT_OUT = Path("outputs/fingpt_sft_lora")
DPO_OUT = Path("outputs/fingpt_dpo_lora")
MERGED_SFT_OUT = Path("outputs/fingpt_sft_merged")  # 可选
MERGED_DPO_OUT = Path("outputs/fingpt_dpo_merged")  # 可选

RAW_DIR.mkdir(parents=True, exist_ok=True)
SFT_DIR.mkdir(parents=True, exist_ok=True)
DPO_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL:", BASE_MODEL)
print("FIN_DATASET:", FIN_DATASET, "split=", FIN_SPLIT)
print("RAW_FILE:", RAW_FILE)
print("SFT_FILE:", SFT_FILE)
print("DPO_FILE:", DPO_FILE)


## 2. 下载 FinGPT 数据并保存为本地 jsonl


In [ ]:
import json
from datasets import load_dataset

ds = load_dataset(FIN_DATASET, split=FIN_SPLIT)
with RAW_FILE.open("w", encoding="utf-8") as f:
    for row in ds:
        f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

print("saved rows:", len(ds))
print("saved to:", RAW_FILE)


## 3. 转换为 MedicalGPT 的 SFT / DPO 数据格式

- `fin_to_sharegpt.py` 产出 SFT 的 `conversations` 格式
- `fin_to_dpo_pairs.py` 产出 DPO 的 `question/response_chosen/response_rejected` 格式


In [ ]:
import subprocess

subprocess.run(
    [
        "python", "fin_to_sharegpt.py",
        "--source_file", str(RAW_FILE),
        "--output_file", str(SFT_FILE),
    ],
    check=True,
)

subprocess.run(
    [
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(RAW_FILE),
        "--output_file", str(DPO_FILE),
        "--seed", "42",
    ],
    check=True,
)

print("SFT converted ->", SFT_FILE)
print("DPO converted ->", DPO_FILE)


In [ ]:
# 抽样查看格式是否符合 docs/datasets.md
import json
from itertools import islice

print("[SFT sample]")
with SFT_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(json.loads(line))

print("\n[DPO sample]")
with DPO_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(json.loads(line))


## 4. SFT 完整训练（LoRA）

这里给出完整训练参数模板（按你的资源调整 batch size / epoch / max length）。


In [ ]:
sft_cmd = [
    "python", "supervised_finetuning.py",
    "--model_name_or_path", BASE_MODEL,
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(SFT_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--num_train_epochs", "3",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "2e-4",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--model_max_length", "1024",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(SFT_OUT),
    "--overwrite_output_dir",
]
print(" ".join(sft_cmd))
subprocess.run(sft_cmd, check=True)


## 5. DPO 训练（LoRA）

DPO 以 SFT 结果作为起点（`--model_name_or_path` 指向 SFT 输出目录）。


In [ ]:
dpo_cmd = [
    "python", "dpo_training.py",
    "--model_name_or_path", str(SFT_OUT),
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(DPO_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "5e-7",
    "--num_train_epochs", "2",
    "--max_length", "1024",
    "--max_prompt_length", "512",
    "--beta", "0.1",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(DPO_OUT),
    "--overwrite_output_dir",
]
print(" ".join(dpo_cmd))
subprocess.run(dpo_cmd, check=True)


## 6. （可选）合并 LoRA 权重

如需导出可直接推理的完整权重，可分别合并 SFT / DPO LoRA。


In [ ]:
merge_sft_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(SFT_OUT),
    "--output_dir", str(MERGED_SFT_OUT),
]
print(" ".join(merge_sft_cmd))
# subprocess.run(merge_sft_cmd, check=True)

merge_dpo_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(DPO_OUT),
    "--output_dir", str(MERGED_DPO_OUT),
]
print(" ".join(merge_dpo_cmd))
# subprocess.run(merge_dpo_cmd, check=True)
